# RAWG API Task — Video Game Data Analysis

Consuming the RAWG API to extract, analyze, and compare video game data using Python.

## Setup — Imports and API Configuration

Importing required libraries and defining the API Key as a local variable.  
**Important:** do not upload your API Key to GitHub.

In [24]:
import requests
import pandas as pd

# Store your API Key as a local variable — do not share or commit this value
API_KEY = "a52a8e28a079470caa05a06f77a5093e"
BASE_URL = "https://api.rawg.io/api"

request_count = 0

def get(endpoint, params=None):
    """Helper to make authenticated GET requests to the RAWG API."""
    global request_count
    if params is None:
        params = {}
    params["key"] = API_KEY
    response = requests.get(f"{BASE_URL}{endpoint}", params=params)
    response.raise_for_status()
    request_count += 1
    return response.json()

def resumen_requests():
    print(f"Total API requests made: {request_count}")

print("Setup complete.")

Setup complete.


## Part A — General Exploration

### A1 — How many games does RAWG have registered in total?

We query the `/games` endpoint and read the `count` field from the response, which represents the total number of games in the database.

In [25]:
data = get("/games")
total_games = data["count"]
print(f"Total number of games registered in RAWG: {total_games:,}")

Total number of games registered in RAWG: 898,502


## Part B — Category Analysis

### B1 — Top 5 highest rated games of all time (by Metacritic)

We query the `/games` endpoint ordering by Metacritic score and display the top 5 results showing name, rating, and metacritic score.

In [26]:
data_b1 = get("/games", params={"ordering": "-metacritic", "page_size": 5})

top5_metacritic = [
    {"name": g["name"], "rating": g["rating"], "metacritic": g["metacritic"]}
    for g in data_b1["results"]
]

df_b1 = pd.DataFrame(top5_metacritic)
df_b1.index += 1
print("Top 5 highest rated games of all time (Metacritic):")
df_b1

Top 5 highest rated games of all time (Metacritic):


,name,rating,metacritic
1,The Legend of Zelda: Ocarina of Time,4.38,99
2,Soulcalibur (1998),0.00,98
3,Soulcalibur,4.38,98
4,Baldur's Gate III,4.44,97
5,Metroid Prime,4.35,97


### B2 — Top 10 best games available on Steam (store_id=1)

We filter by Steam (`stores=1`) and order by rating to get the 10 best games, showing name, rating, and metacritic score.

In [27]:
data_b2 = get("/games", params={"stores": 1, "ordering": "-rating", "page_size": 10})

top10_steam = [
    {"name": g["name"], "rating": g["rating"], "metacritic": g["metacritic"]}
    for g in data_b2["results"]
]

df_b2 = pd.DataFrame(top10_steam)
df_b2.index += 1
print("Top 10 best games available on Steam:")
df_b2

Top 10 best games available on Steam:


,name,rating,metacritic
1,"Warhammer 40,000: Dawn of War - Definitive Edi...",4.83,NaN
2,No Case Should Remain Unsolved,4.83,NaN
3,The Witcher 3: Wild Hunt – Blood and Wine,4.81,92.0
4,The Witcher 3 Wild Hunt - Complete Edition,4.80,92.0
5,The Witcher 3: Wild Hunt – Hearts of Stone,4.76,90.0
6,Persona 5 Royal,4.75,94.0
7,Guilty Parade,4.71,NaN
8,Cyberpunk 2077: Phantom Liberty,4.71,NaN
9,The Binding of Isaac: Repentance,4.69,NaN
10,Red Matter 2,4.67,NaN


## Part C — Comparisons

### C1 — Top 5 games on PC (platform_id=4) vs Top 5 on PS5 (platform_id=187)

We query both platforms separately, compare their average ratings, and determine which platform has the highest rated games.

In [28]:
data_pc = get("/games", params={"platforms": 4, "ordering": "-rating", "page_size": 5})
data_ps5 = get("/games", params={"platforms": 187, "ordering": "-rating", "page_size": 5})

df_pc = pd.DataFrame([
    {"name": g["name"], "rating": g["rating"], "metacritic": g["metacritic"]}
    for g in data_pc["results"]
])
df_ps5 = pd.DataFrame([
    {"name": g["name"], "rating": g["rating"], "metacritic": g["metacritic"]}
    for g in data_ps5["results"]
])

df_pc.index += 1
df_ps5.index += 1

avg_pc = df_pc["rating"].mean()
avg_ps5 = df_ps5["rating"].mean()

print("Top 5 games on PC:")
print(df_pc.to_string())
print(f"\nAverage rating on PC: {avg_pc:.2f}")

print("\nTop 5 games on PS5:")
print(df_ps5.to_string())
print(f"\nAverage rating on PS5: {avg_ps5:.2f}")

winner = "PC" if avg_pc > avg_ps5 else "PS5"
print(f"\nConclusion: {winner} has the highest rated games with an average rating of {max(avg_pc, avg_ps5):.2f}.")

Top 5 games on PC:
                                                 name  rating  metacritic
1                                The Elder Scrolls VI    4.86         NaN
2  Warhammer 40,000: Dawn of War - Definitive Edition    4.83         NaN
3                      No Case Should Remain Unsolved    4.83         NaN
4           The Witcher 3: Wild Hunt – Blood and Wine    4.81        92.0
5          The Witcher 3 Wild Hunt - Complete Edition    4.80        92.0

Average rating on PC: 4.83

Top 5 games on PS5:
                                         name  rating  metacritic
1  The Witcher 3 Wild Hunt - Complete Edition    4.80        92.0
2                             Persona 5 Royal    4.75        94.0
3             Cyberpunk 2077: Phantom Liberty    4.71         NaN
4            The Binding of Isaac: Repentance    4.69         NaN
5                       The Last of Us Part I    4.67         NaN

Average rating on PS5: 4.72

Conclusion: PC has the highest rated games with an average rat

### C2 — Comparison table for 3 famous games

We search for 3 iconic games and build a comparison table showing name, rating, metacritic, genres, and platforms.

In [29]:
famous_games = ["The Witcher 3: Wild Hunt", "Red Dead Redemption 2", "Grand Theft Auto V"]

rows = []
for title in famous_games:
    result = get("/games", params={"search": title, "page_size": 1})
    if result["results"]:
        g = result["results"][0]
        rows.append({
            "name": g["name"],
            "rating": g["rating"],
            "metacritic": g["metacritic"],
            "genres": ", ".join(genre["name"] for genre in g.get("genres", [])),
            "platforms": ", ".join(p["platform"]["name"] for p in g.get("platforms", []))
        })

df_c2 = pd.DataFrame(rows)
df_c2.index += 1
print("Comparison table — 3 famous games:")
df_c2

Comparison table — 3 famous games:


,name,rating,metacritic,genres,platforms
1,The Witcher 3: Wild Hunt,4.64,92,"Action, RPG","PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."
2,Red Dead Redemption 2,4.59,96,Action,"PC, Xbox One, PlayStation 4"
3,Grand Theft Auto V,4.47,92,Action,"PC, PlayStation 5, Xbox One, PlayStation 4, Xb..."


### C3 — Average rating by genre (4 genres)

We query the top 5 games for each of 4 different genres, calculate the average user rating per genre, and determine which genre produces the best games.

In [30]:
# Genre IDs: action=4, RPG=5, strategy=10, shooter=2
genres = {"Action": 4, "RPG": 5, "Strategy": 10, "Shooter": 2}

genre_rows = []
for genre_name, genre_id in genres.items():
    data_g = get("/games", params={"genres": genre_id, "ordering": "-rating", "page_size": 5})
    ratings = [g["rating"] for g in data_g["results"]]
    avg = sum(ratings) / len(ratings) if ratings else 0
    genre_rows.append({"genre": genre_name, "avg_rating": round(avg, 2)})

df_c3 = pd.DataFrame(genre_rows).sort_values("avg_rating", ascending=False)
df_c3.index = range(1, len(df_c3) + 1)

print("Average rating by genre (top 5 games each):")
print(df_c3.to_string())
best_genre = df_c3.iloc[0]["genre"]
print(f"\nConclusion: '{best_genre}' produces the best rated games according to users.")

Average rating by genre (top 5 games each):
      genre  avg_rating
1       RPG        4.80
2    Action        4.77
3  Strategy        4.71
4   Shooter        4.68

Conclusion: 'RPG' produces the best rated games according to users.


### C4 — Best games from 3 different years

We compare the best games released in 2015, 2018, and 2022, calculating the average Metacritic score per year to determine which year had the highest rated releases.

In [31]:
years = [2015, 2018, 2022]
year_rows = []

for year in years:
    data_y = get("/games", params={
        "dates": f"{year}-01-01,{year}-12-31",
        "ordering": "-metacritic",
        "page_size": 5
    })
    scores = [g["metacritic"] for g in data_y["results"] if g["metacritic"]]
    avg = sum(scores) / len(scores) if scores else 0
    year_rows.append({"year": year, "avg_metacritic": round(avg, 2)})

df_c4 = pd.DataFrame(year_rows).sort_values("avg_metacritic", ascending=False)
df_c4.index = range(1, len(df_c4) + 1)

print("Average Metacritic score by year (top 5 games each):")
print(df_c4.to_string())
best_year = df_c4.iloc[0]["year"]
print(f"\nConclusion: {best_year} had the highest average Metacritic score ({df_c4.iloc[0]['avg_metacritic']}).")

Average Metacritic score by year (top 5 games each):
   year  avg_metacritic
1  2018            93.6
2  2015            93.2
3  2022            90.2

Conclusion: 2018.0 had the highest average Metacritic score (93.6).


### C5 — Export top 20 games of all time to CSV

We fetch the top 20 games ordered by Metacritic score and export them to `output/top20_rawg.csv` with columns: `name`, `rating`, `metacritic`, `release_date`, `main_genre`. We then display the first 5 rows.

In [32]:
data_c5 = get("/games", params={"ordering": "-metacritic", "page_size": 20})

top20 = []
for g in data_c5["results"]:
    main_genre = g["genres"][0]["name"] if g.get("genres") else "N/A"
    top20.append({
        "name": g["name"],
        "rating": g["rating"],
        "metacritic": g["metacritic"],
        "release_date": g["released"],
        "main_genre": main_genre
    })

df_c5 = pd.DataFrame(top20)
df_c5.index += 1

output_path = "output/top20_rawg.csv"
df_c5.to_csv(output_path, index=False)
print(f"CSV exported to {output_path}")
print("\nFirst 5 rows:")
df_c5.head()

CSV exported to output/top20_rawg.csv

First 5 rows:


,name,rating,metacritic,release_date,main_genre
1,The Legend of Zelda: Ocarina of Time,4.38,99,1998-11-21,Action
2,Soulcalibur (1998),0.00,98,1998-07-30,Fighting
3,Soulcalibur,4.38,98,1998-07-30,Action
4,Baldur's Gate III,4.44,97,2023-08-03,Adventure
5,Metroid Prime,4.35,97,2002-11-17,Action


## Part D — Insights & Conclusions

### D1 — Personal Conclusions

**What was the most interesting thing you found in the data?**

The most striking finding was the difference between Metacritic scores and user ratings, I mean, you can just check all the top 5 games on PC, PS5, Steam have a much better rating than the top score of Metacritic (The Legend of Zelda: Ocarina of Time).Overall, highly valued games by Metacritic can't compete against the top rated games in all these platforms, they are even lower than the average ratings we saw on C4. I interpret this as a difference between what factors a critic value in a game, vs what player value; critics will probably tend to reward technical ambition and polish, while users seem to care more about replayability, community, and overall enjoyment. It made me realize that using a single score to rank games is fundamentally incomplete.

**Which genre or platform surprised you the most and why?**

Strategy games surprised me the most. Going in, I expected Action or RPG to dominate in user ratings since those genres have the biggest mainstream audiences (and me personally find the most appeal in these genres) and the most passionate fanbases. But strategy titles have a ver respectable rating, and in some cases outperformed them. This probably reflects the fact that strategy games attract a smaller but extremely dedicated player base that rates games very carefully, which pushes average scores up. Though I kinda expected shooters to be lower than the other genres, as (at least in my opinion) they tend to get repetitive and loose appeal after playing for a long time, which is what is happening to Call of Duty nowadays.

**What other question would you ask this API if you had more time?**

I would look at how a game's rating evolves in relation to its number of reviews. Specifically, I'd want to know whether games with very few ratings tend to score higher or lower than those with thousands of reviews, or maybe even check how ratings evolve over time, I mean maybe ratings are pretty good when the game as new, because of the hype, but then ratings lower as people have more time to completely experience the game. Finally I think it would be good to see the deviation between ratings, like to check if a game has been review bombed, and its rating should be higher, I've seen that type of stuff happen on IMDb, where the ratings from movies or episodes can be concentrated in 10 and 1, the 1s being people who just wanted to lower the average rating just because they are haters.

**How many requests did you use in total?**

*(see cell below — run it after executing the full notebook)*

In [33]:
resumen_requests()

Total API requests made: 16
